In [1]:
# Install first:
# pip install dukascopy-python pandas numpy tqdm

from pathlib import Path
from datetime import datetime
import time

import numpy as np
import pandas as pd
from tqdm import tqdm

import dukascopy_python
from dukascopy_python.instruments import (
    INSTRUMENT_FX_MAJORS_USD_JPY
)


# -----------------------------------
# Settings
# -----------------------------------

START_DATE = datetime(1998, 12, 14)

# Use 2 July as the end boundary so that
# all data from 1 July 2026 is included.
END_DATE = datetime(2026, 7, 2)

OUTPUT_DIR = Path("usdjpy_hourly_data")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

BID_FILE = OUTPUT_DIR / "usdjpy_hourly_bid.csv"
ASK_FILE = OUTPUT_DIR / "usdjpy_hourly_ask.csv"
MID_FILE = OUTPUT_DIR / "usdjpy_hourly_mid.csv"


# -----------------------------------
# Download function
# -----------------------------------

def download_data(start, end, offer_side):
    """
    Download hourly USD/JPY OHLC data
    for a specified date range and quote side.
    """

    return dukascopy_python.fetch(
        INSTRUMENT_FX_MAJORS_USD_JPY,
        dukascopy_python.INTERVAL_HOUR_1,
        offer_side,
        start,
        end,
    )


# -----------------------------------
# Download yearly chunks
# -----------------------------------

bid_data = []
ask_data = []

for year in tqdm(
    range(1998, 2027),
    desc="Downloading yearly data"
):

    chunk_start = max(
        START_DATE,
        datetime(year, 1, 1)
    )

    chunk_end = min(
        END_DATE,
        datetime(year + 1, 1, 1)
    )

    if chunk_start >= chunk_end:
        continue

    print(
        f"\nDownloading "
        f"{chunk_start.date()} to {chunk_end.date()}"
    )

    try:
        # Download BID candles
        bid_chunk = download_data(
            chunk_start,
            chunk_end,
            dukascopy_python.OFFER_SIDE_BID
        )

        # Download ASK candles
        ask_chunk = download_data(
            chunk_start,
            chunk_end,
            dukascopy_python.OFFER_SIDE_ASK
        )

        if bid_chunk is not None and not bid_chunk.empty:
            bid_data.append(bid_chunk)

        if ask_chunk is not None and not ask_chunk.empty:
            ask_data.append(ask_chunk)

        # Small delay between yearly requests
        time.sleep(1)

    except Exception as error:
        print(f"Failed for {year}: {error}")
        continue


# -----------------------------------
# Check that data was downloaded
# -----------------------------------

if not bid_data:
    raise RuntimeError("No BID data was downloaded.")

if not ask_data:
    raise RuntimeError("No ASK data was downloaded.")


# -----------------------------------
# Combine BID and ASK data
# -----------------------------------

bid = pd.concat(bid_data)
ask = pd.concat(ask_data)

# Convert timestamps to UTC
bid.index = pd.to_datetime(bid.index, utc=True)
ask.index = pd.to_datetime(ask.index, utc=True)

# Sort chronologically
bid = bid.sort_index()
ask = ask.sort_index()

# Remove duplicate timestamps
bid = bid[~bid.index.duplicated(keep="first")]
ask = ask[~ask.index.duplicated(keep="first")]

# Define exact UTC date boundaries
start_timestamp = pd.Timestamp(
    START_DATE,
    tz="UTC"
)

end_timestamp = pd.Timestamp(
    END_DATE,
    tz="UTC"
)

# Restrict data to the requested date range
bid = bid.loc[
    (bid.index >= start_timestamp) &
    (bid.index < end_timestamp)
]

ask = ask.loc[
    (ask.index >= start_timestamp) &
    (ask.index < end_timestamp)
]

# Save the separate BID and ASK files
bid.to_csv(BID_FILE)
ask.to_csv(ASK_FILE)


# -----------------------------------
# Match BID and ASK timestamps
# -----------------------------------

common_index = bid.index.intersection(ask.index)

bid = bid.loc[common_index]
ask = ask.loc[common_index]


# -----------------------------------
# Calculate midpoint OHLC prices
# -----------------------------------

mid = pd.DataFrame(index=common_index)

for column in ["open", "high", "low", "close"]:
    mid[f"mid_{column}"] = (
        bid[column] + ask[column]
    ) / 2


# Calculate the closing spread
mid["spread_close"] = (
    ask["close"] - bid["close"]
)


# Calculate hourly log returns
mid["log_return"] = np.log(
    mid["mid_close"] / mid["mid_close"].shift(1)
)

# Remove the first observation,
# which has no previous price
mid = mid.dropna(
    subset=["mid_close", "log_return"]
)


# -----------------------------------
# Save midpoint data
# -----------------------------------

mid.to_csv(MID_FILE)


# -----------------------------------
# Display output information
# -----------------------------------

print("\nFinished.")
print(f"Number of observations: {len(mid):,}")
print(f"First timestamp: {mid.index.min()}")
print(f"Last timestamp:  {mid.index.max()}")
print(f"BID file: {BID_FILE.resolve()}")
print(f"ASK file: {ASK_FILE.resolve()}")
print(f"MID file: {MID_FILE.resolve()}")


Finished.
Number of observations: 149,842
First timestamp: 2003-05-04 20:00:00+00:00
Last timestamp:  2026-07-01 23:00:00+00:00
BID file: C:\Users\Rajiv Nawal\OneDrive\Documents\GITREPOS\HSBC-AN-RN\CODEWORK\Linus' Task Re-Do\Data\usdjpy_hourly_data\usdjpy_hourly_bid.csv
ASK file: C:\Users\Rajiv Nawal\OneDrive\Documents\GITREPOS\HSBC-AN-RN\CODEWORK\Linus' Task Re-Do\Data\usdjpy_hourly_data\usdjpy_hourly_ask.csv
MID file: C:\Users\Rajiv Nawal\OneDrive\Documents\GITREPOS\HSBC-AN-RN\CODEWORK\Linus' Task Re-Do\Data\usdjpy_hourly_data\usdjpy_hourly_mid.csv
